In [ ]:
# Split Data

import os, shutil, random

src_root = "treeoflife_sanity_sample"     # class_name/*.jpg
dst_root = "treeoflife_sanity_sample_split"
val_frac = 0.2

os.makedirs(dst_root, exist_ok=True)
for cls in os.listdir(src_root):
    files = os.listdir(f"{src_root}/{cls}")
    random.shuffle(files)
    n_val = int(len(files) * val_frac)
    for split, split_files in [("val", files[:n_val]), ("train", files[n_val:])]:
        out_dir = f"{dst_root}/{split}/{cls}"
        os.makedirs(out_dir, exist_ok=True)
        for f in split_files:
            shutil.copy(f"{src_root}/{cls}/{f}", f"{out_dir}/{f}")

In [2]:
# Create Label File

import os

classes = sorted(os.listdir("treeoflife_sanity_sample_split/train"))
with open("treeoflife_sanity_sample_split.txt", "w") as f:
    f.write("\n".join(classes))

In [ ]:
# Create Concept Set

import conceptset_utils

classes = ["Anatidae", "Canidae", "Felidae", "Nymphalidae", "Pinaceae"]

with open("concepts/treeoflife_raw.txt") as f:
    concepts = [line.strip() for line in f if line.strip()]

# 1. Remove concepts longer than MAX_LEN characters (default 30)
concepts = conceptset_utils.remove_too_long(concepts, max_len=30, print_prob=1)

# 2. Remove concepts too similar to the class names themselves
#    (e.g. "bird" would be removed if too close to "Anatidae")
CLASS_SIM_CUTOFF = 0.85
concepts = conceptset_utils.filter_too_similar_to_cls(
    concepts, classes, sim_cutoff=CLASS_SIM_CUTOFF, print_prob=1
)

# 3. Remove near-duplicate concepts among themselves
#    (e.g. "a forest" appearing for both Canidae and Pinaceae collapses to one)
OTHER_SIM_CUTOFF = 0.9
concepts = conceptset_utils.filter_too_similar(
    concepts, sim_cutoff=OTHER_SIM_CUTOFF, print_prob=1
)

with open("concepts/treeoflife_filtered.txt", "w") as f:
    f.write("\n".join(concepts))